In [1]:
# conda env 'vis2_clone'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet
from src.Retrieval import Retrieval

plt.style.use('seaborn-v0_8-paper')

In [3]:
es_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V11'
exset = ExperimentSet(es_dir)
exset.set_retrieval_list()
exset.load_retrievals(iteration=-1, extinction_th=[0.0, np.inf])

In [4]:
cvm = exset.get_retrieval('default').cvm.drop_vars(['delx', 'dely'])
cvm['cloud_top_heights'] = cvm.z.where(cvm.cloud_volume_mask == 1).max(dim="z")
cvm['distance_to_cloud_top'] = cvm.z - cvm.cloud_top_heights
cvm['cloud_base_heights'] = cvm.z.where(cvm.cloud_volume_mask == 1).min(dim="z")
cvm['distance_to_cloud_base'] = cvm.z - cvm.cloud_base_heights
distance_to_cloud = cvm.distance_to_cloud_base.clip(max=0) + cvm.distance_to_cloud_top.clip(min=0)
cvm["distance_to_cloud"] = distance_to_cloud
cvm = cvm.drop_vars(['cloud_top_heights', 'cloud_base_heights'])
cvm = cvm.to_dataframe()
df_list = []

for r in list(exset.retrievals.values()):
    tmp_ext = r.solvers_retrieval[0.67].medium['cloud'].extinction.copy(deep=True)
    df_tmp = tmp_ext.drop_vars(['reff', 'veff']).to_dataframe()
    df_tmp = df_tmp.join(cvm)
    df_tmp['source'] = r.exp_name
    df_list.append(df_tmp)

df = pd.concat(df_list)
# df = df.pivot(columns='source', values='extinction')
# df = df.join(exset.get_retrieval('default').cvm.drop_vars(['delx', 'dely']).to_dataframe())

df = df.reset_index()
# df['x'] *= 1000
# df['y'] *= 1000
# df['z'] *= 1000

bins = np.arange(-50, 1451, 100)
df['altitude_bin'] = pd.cut(df['z'], bins=bins)

dtc_bins = np.arange(-0.45, 1.051, 0.1)
df['distance_to_cloud_bin'] = pd.cut(df['distance_to_cloud'], bins=dtc_bins)

df['isCloud'] = df['extinction'] > 0.0

cvm = cvm.reset_index()

# All interior (remove outer columns)

In [5]:
internal_df = df[
    (df['x'] != 0.0)
    & (df['y'] != 0.0)
    & (df['x'] != 17.6)
    & (df['y'] != 17.6)
]
cvm_by_region = internal_df.pivot_table(columns='source', index=['lcl', 'cth', 'sc'], values='isCloud', aggfunc='sum')

internal_cvm = cvm[
    (cvm['x'] != 0.0)
    & (cvm['y'] != 0.0)
    & (cvm['x'] != 17.6)
    & (cvm['y'] != 17.6)
]
cvm_by_region['total number grid points'] = internal_cvm[['lcl', 'cth', 'sc']].value_counts()

cvm_by_region.loc[:,['total number grid points', 'default', 'unconstrained']]

source      total number grid points  default  unconstrained
lcl cth sc                                                  
0   0   0                         19        0              0
    1   0                       1997        0              0
        1                      13860        0           1172
1   0   0                       2865        0              0
        1                      27321        0           2240
    1   0                       1507        0              0
        1                      11966     3080           2416

# z = 1.4 km

In [ ]:
internal_df = df[
    (df['x'] != 0.0)
    & (df['y'] != 0.0)
    & (df['x'] != 17.6)
    & (df['y'] != 17.6)
    & (df['z'] == 1.4)
]
cvm_by_region = internal_df.pivot_table(columns='source', index=['lcl', 'cth', 'sc'], values='isCloud', aggfunc='sum')

internal_cvm = cvm[
    (cvm['x'] != 0.0)
    & (cvm['y'] != 0.0)
    & (cvm['x'] != 17.6)
    & (cvm['y'] != 17.6)
    & (cvm['z'] == 1.4)
]
cvm_by_region['total number grid points'] = internal_cvm[['lcl', 'cth', 'sc']].value_counts()

cvm_by_region.loc[:,['default', 'unconstrained', 'total number grid points']]